# Step 1：带间隙约束的需求加权部署

按 **研究问题 → 方法 → 评价 → 优化参照 → 结果 → 局限** 阅读已经保存的结果。核心计算在 `src/udc/`，这里只调用包内函数并读取轨迹。

## 研究问题

在固定 Guide 数量下，怎样提高热点一侧的需求覆盖，同时保持最大弧间隙、最小间距和可达性。人员权重均匀并不使 Gaussian 曲线密度均匀。上一轮凸曲线上，浓度调节没有触发，因此当前要评价的是间隙投影本身。

## 方法

`projected_mass` 是带覆盖间隙约束的需求加权边界部署：在原始需求密度上做周期 CVT，每一步把弧间隙投影到 $[d_{\min}, H_p]$，并保留整体相位。RA-ABDC 使用同一次投影；只有欧氏间距仍然失败时才减半浓度。那一步在已有实验里没有发生。

数值参照用同一目标和同一积分，从 Equal Arc 与 `projected_mass` 出发做 SLSQP。它只产生目标位置，不是完整任务的最优解。

## 评价

全曲线代价是原始 Gaussian 需求下的均方弧长距离，单位 m²，越小越好。热点超额是冻结需求减去背景，核与带宽不变。局部超额达到该点需求一半的弧段不依赖 Guide。没有超额时，热点指标不适用。目标位置和实际位置分开。实际点离开曲线超过容差时单独标记，不用投影把偏离藏起来。

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

from udc.analysis.reanalysis import reanalyze

ROOT = Path.cwd() if (Path.cwd() / 'runs' / 'ablation' / 'projection_matrix').exists() else Path.cwd().parent
summary_path = ROOT / 'runs' / 'ablation' / 'excess_reanalysis' / 'excess_summary.json'
if not summary_path.exists():
    reanalyze(ROOT / 'runs' / 'ablation' / 'projection_matrix' / 'comparison.json',
              ROOT / 'runs' / 'ablation' / 'excess_reanalysis')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('uniform not applicable', summary['uniform_marked_not_applicable'])
print('projected minus equal', summary['projected_minus_equal_demand_cost'])
print('rescued ratio of means', summary['rescued']['demand_retention_ratio_of_means'])
for name in ('circle_strong_near_seed0.png', 'ellipse_strong_near_seed0.png'):
    display(Image(filename=str(ROOT / 'runs' / 'ablation' / 'excess_reanalysis' / 'figures' / name)))

## 优化参照

12 个贴近数量下界的场景只做规划比较。结果在 `runs/ablation/reference_seed0` 和 `reference_seed1`。积分分辨率检查在 `runs/ablation/reference_sensitivity`。参照相对 `projected_mass` 的改进约为 0.0002–0.005 m²，小于强热点上投影相对 Equal Arc 的改进，并且不随积分点数从 1024 增到 4096 而改变量级。

## 结果

90 对里，`projected_mass` 的全曲线规划代价都低于 Equal Arc，均值差 −0.00513 m²。Mass CVT 已经可行的 62 对上，二者几乎相同。28 个原先不可行的任务被投影修复；三个规划代价的均值是 0.275789、0.282797、0.287497，投影留下约 40.1% 的无约束代价改进。这只描述这 28 个任务，不是全体任务的性能涨幅。

多初值参照能再降一点代价，尤其是圆形强热点，但没有找到另一类明显更好的可行方案。若干次迭代达到上限，独立检查仍然可行；上限用尽不是不可行。

## 局限

数量界只在“目标位于这一条曲线、弦长不超过弧长”时是必要条件。投影保证的是弧间隙盒子里的数值可行点，不保证原始目标下降、全局最优或欧氏间距。可达和执行安全仍由原有导航与线段复核负责。浓度调节没有被检验。五个开发 seed 不能支持换一组正式 seed 之前的一般结论。本轮之后不再自动扩大实验。